## Computing isogeny cycles

The following function takes as input:
* A pair of Weierstrass coefficients (f,g), that represents an elliptic curve which is assumed to be supersingular.
* A prime $p>3$
* A prime $l \neq p$, for which $-4p$ is a quadratic residue.
It will return a list of pairs $(f_i,g_i)$ that represent supersingular elliptic curves,
with the property that two curves are related by any isogeny of degree $\ell$ iff they are adjacent
in the list.

In [80]:
def j_to_fg(j:int,char = 0):
    if j == 0:
        return (0,1)
    elif j == 1728 or char>0 and (j-1728)%char == 0:
        return (1,0)
    else:
        f = -3*j*(j-1728)
        g = 2*j*((j-1728)**2)
        if char == 0:
            return (f,g)
        else:
            return (f % char, g% char)
        
def fg_to_j(fg:tuple[int,int],char =0):
    f,g = fg
    if f == 0 or char>0 and f%char ==0:
        return 0
    elif g == 0 or char>0 and g % char ==0:
        return 1728
    else:
        f3 = 4*(f**3)
        jnum = 1728 *f3
        jden = f3+27*(g**2)
        if char == 0:
            if jden == 0:
                raise ZeroDivisionError('Singular curve')
            elif jnum % jden == 0:
                return jnum//jden
            else:
                return jnum/jden
        else:
            jnum = jnum % char
            jden = jden % char
            if jden == 0:
                raise ZeroDivisionError('Singular curve')
            jdeninv = pow(jden,-1,char)
            return (jnum*jdeninv)%char

def fg_to_sig(fg,p):
    j = fg_to_j(fg,p)
    f,g = fg
    f,g = int(f),int(g)
    if g %p == 0:
        return (j%p,quad_rec(f,p))
    else:
        return (j%p,quad_rec(g,p))


def ss_codomains_fp(fg,p,l):
    n = Mod(-p,l).multiplicative_order()
    k = GF(p**(2*n))
    E = EllipticCurve(k,fg)
    e1,e2 = E.torsion_basis(l)
    gens = [e1]+[e2+a*e1 for a in range(l)]
    cod_coefs = [EllipticCurveIsogeny(E,pt).codomain().a_invariants()[3:] for pt in gens]
    coefs_fp = []
    for fg1 in cod_coefs:
        f1,g1 = fg1
        if f1**p== f1 and g1**p == g1:
            coefs_fp.append(fg1)
    return coefs_fp 

def ss_fp_cycle(fg,p,l):
    if p == l:
        raise ValueError('p and l must be different')
    if quad_rec(-p,l)!= 1:
        raise ValueError('No supersingular isogenies in this degree')
    E0 = EllipticCurve(GF(p),fg)
    if E0.cardinality()!=p+1:
        raise ValueError('Curve is not supersingular')
    cycle = [fg]
    nextbatch= ss_codomains_fp(fg,p,l)
    if len(nextbatch)==0:
        raise ValueError('No isogenies found')
    fg1 = nextbatch[0]
    E1 = EllipticCurve(GF(p),fg1)
    while not (E0.is_isomorphic(E1)):
        Eprev = EllipticCurve(GF(p),cycle[-1])
        e1coefs = E1.a_invariants()[3:]
        cycle.append(e1coefs)
        nextbatch = ss_codomains_fp(e1coefs,p,l)
        E2test = EllipticCurve(GF(p),nextbatch[0])
        if not Eprev.is_isomorphic(E2test):
            E1 = E2test
        else:
            E1 = EllipticCurve(GF(p),nextbatch[1])
    return cycle
    
def fg_to_isocycle_sigs(fg,p,l):
    cycle = ss_fp_cycle(fg,p,l)
    return [fg_to_sig(fg,p) for fg in cycle]

def reverse_coset(coset):
    return coset[:1]+coset[:0:-1]

def ss_fp3m8_cosets(pl):
    p, l = pl
    if p % 8 != 3:
        raise ValueError('p must be congruent to 3 mod 8')
    if quad_rec(-p,l)!=1:
        raise ValueError('No supersingular Fp isogenies in this degree')
    ec0 = EllipticCurve(GF(p),(-1,0))
    t0,t1 = ec0.torsion_basis(2)
    fg0s = [EllipticCurveIsogeny(ec0,pt).codomain().a_invariants()[3:] for pt in [t0,t1,t0+t1]]
    fg0s.sort(key = lambda x:(1728-fg_to_j(x,p))%p)
    cycles =  [fg_to_isocycle_sigs(fg,p,l) for fg in fg0s]
    return cycles

def ecfp_2des_sigs(fg,p):
    ec = EllipticCurve(GF(p),fg)
    t0,t1 = ec.torsion_basis(2)
    fgs = [EllipticCurveIsogeny(ec,pt).codomain().a_invariants()[3:] for pt in [t0,t1,t0+t1]]
    return [fg_to_sig(fg,p) for fg in fgs]

def ss3m8_2grs(p,l):
    if p % 8 != 3:
        raise ValueError('p must be 3 mod 8')
    if quad_rec(-p,l)!= 1:
        raise ValueError('no isogenies of degree l')
    fgcyc1 = ss_fp_cycle((-1,0),p,l)
    return [ecfp_2des_sigs(fg,p) for fg in fgcyc1]

def orient_cs(cosets,transverse):
    newcosets = []
    for c in cosets:
        if c[1] in transverse:
            newcosets.append(c)
        else:
            newcosets.append(reverse_coset(c))
    return newcosets

def ss_fp3m8_cosets_oriented(pl):
    p,l = pl
    cosets = ss_fp3m8_cosets(pl)
    transv = ss3m8_2grs(p,l)[1]
    return orient_cs(cosets,transv)

In [ ]:
import os
os.chdir('../code')

In [20]:
from nt import quad_rec, primesBetween

In [82]:
ss_fp3m8_cosets_oriented((547,13))

[[(87, 1), (167, 1), (167, -1)],
 [(321, -1), (52, -1), (310, 1)],
 [(321, 1), (310, -1), (52, 1)]]

In [16]:
def ss_iso_deg(p,l):
    if quad_rec(-p,l)!= 1:
        return 0
    else:
        return 2*(Mod(-p,l).multiplicative_order())

6

In [3]:
E = EllipticCurve(k,(1,0))

In [21]:
qf0 = BQFClassGroup(-4*307).gens()[0]

In [43]:
qf0.form().polynomial().coefficients()

[7, 2, 44]

In [45]:
[ss_iso_deg(307,l) for l in [3,5,7,11,13,17]]

[0, 0, 2, 2, 0, 4]

In [46]:
primes3m8 = [p for p in primesBetween(256,2056) if p % 8 == 3]

In [49]:
pr3m8classgrgens = {p:BQFClassGroup(-4*p).gens() for p in primes3m8}

In [51]:
p3m8noncycl = {p:pr3m8classgrgens[p] for p in pr3m8classgrgens if len(pr3m8classgrgens[p])>1}

In [ ]:
classgens_all = {p:BQFClassGroup(-4*p).gens() for p in primesBetween(256,2048)}
classgens_x = {p:classgens_all[p] for p in classgens_all if len(classgens_all[p])>1}

In [58]:
len(classgens_x)

5

In [59]:
classgens_x

{307: (Class of 7*x^2 + 2*x*y + 44*y^2, Class of 11*x^2 + 2*x*y + 28*y^2),
 547: (Class of 11*x^2 + 10*x*y + 52*y^2, Class of 13*x^2 + 10*x*y + 44*y^2),
 1187: (Class of 3*x^2 + 2*x*y + 396*y^2, Class of 11*x^2 + 2*x*y + 108*y^2),
 1259: (Class of 3*x^2 + 2*x*y + 420*y^2, Class of 15*x^2 + 2*x*y + 84*y^2),
 1931: (Class of 37*x^2 - 34*x*y + 60*y^2, Class of 15*x^2 + 14*x*y + 132*y^2)}

In [64]:
classgens_x[1187][1].order()

3

In [66]:
{p:[g.order() for g in classgens_x[p]] for p in classgens_x}

{307: [3, 3], 547: [3, 3], 1187: [9, 3], 1259: [15, 3], 1931: [21, 3]}

In [67]:
[p for p in classgens_x]

[307, 547, 1187, 1259, 1931]